# Phase 4: Building Localization / Segmentation Model

## 1. Objective

This notebook builds and trains the building segmentation model — a U-Net convolutional neural network that takes a pre-disaster satellite image and predicts a binary mask showing where every building is located. This is the first deep learning model in the project, and the foundation Phase 5 will build on (using the buildings this model finds to guide damage classification on post-disaster imagery).

**Starting point:** Phase 3 finalized the preprocessing pipeline — a leakage-free 80/20 train/validation split (by location ID), a `SegmentationDataset` PyTorch class supplying resized (512×512) pre-disaster image + binary target mask pairs, and a training-only augmentation strategy. This phase uses that pipeline directly.

## 2. What This Notebook Covers

1. **4.1** — Concept: CNN & segmentation fundamentals (convolution, pooling, encoder-decoder, skip connections)
2. **4.2** — Building the U-Net architecture in PyTorch
3. **4.3** — Loss function, optimizer, and the training loop
4. **4.4** — Training on Colab/Kaggle GPU (per the project's hardware-scoping decision)
5. **4.5** — Training multiple candidate segmentation architectures (per the multi-model comparison decision)
6. **4.6** — Monitoring training/validation loss and overfitting
7. **4.7** — Selecting the best segmentation model
8. **4.8** — Saving trained model weights
9. **4.9** — Visual sanity check on validation images

## 3. Honest Scope Note

Per `docs/scope_and_assumptions.md`, this model is trained on a deliberately scoped subset (3 disaster types, 512×512 resized imagery) appropriate for solo, single-machine development — demonstrating the correct methodology for this task, not an infrastructure-scale production system.

## 4.1 Concept: CNNs & Segmentation Fundamentals

Before writing any model code, this section documents the conceptual foundation for the U-Net segmentation model — the first deep learning architecture built in this project.

### Why not a standard neural network?
A normal densely-connected neural network would require flattening an image into one long list of numbers, discarding spatial structure (which pixels are physically near each other) and forcing the model to learn every pattern separately for every position in the image. CNNs solve this by reusing small, learned pattern-detectors across the entire image.

### Convolution
A small matrix called a **kernel** (or filter) slides across small regions of the image. At each region, it multiplies the region's pixel values by the kernel's corresponding values, sums the result into a single number, and moves to the next position. Repeating this across the whole image produces a **feature map** — a new grid highlighting where a specific pattern (e.g. an edge, a corner) appears.

The kernel's values start random and are refined through training via backpropagation — the same correction process used throughout this project (a wrong prediction leads to a small adjustment, repeated across many examples until the kernel reliably detects a meaningful pattern). Early layers tend to learn simple patterns (edges, color transitions); deeper layers combine these into more complex ones (shapes, textures, building-like structures).

### Pooling
**Max pooling** reduces a feature map's spatial size by keeping only the strongest (maximum) value within small regions (e.g. 2×2 blocks). This reduces computation for deeper layers and makes the model more tolerant of small positional shifts in where a feature appears.

### The Encoder
Stacking several rounds of (convolution → pooling) produces an **encoder**: the image shrinks in width/height while growing richer in learned features at each stage. By the end, the network has a compressed, information-dense understanding of the image's content — but has lost precise pixel-location detail along the way.

### The Decoder
Segmentation requires a full-size output mask (a building/not-building prediction for every pixel), not just one label. A **decoder** reverses the encoder's shrinking process, gradually upsampling the compressed representation back toward the original image size.

### Skip Connections (U-Net's defining feature)
The decoder alone, working only from the encoder's final, heavily-compressed output, would produce an imprecise, blurry mask — much of the fine spatial detail was discarded during encoding and can't be fully reconstructed from a small compressed representation alone.

**U-Net's solution:** at each stage of the encoder, before that stage's output is shrunk further, a copy of its feature map is passed directly across to the *matching* stage of the decoder — in parallel with the main compressed path. This gives the decoder access to both the high-level understanding (from the deep path) and precise spatial detail (from the earlier, less-shrunk feature maps), combining them to produce an accurate final mask.

This narrowing-then-widening shape, with cross-connections linking matching levels, is the origin of the name "U-Net" — the architecture diagram forms a U shape.

## 4.2 Building the U-Net Architecture

**Task:** Translate the concepts from 4.1 into a working PyTorch model — an encoder-decoder CNN with skip connections that takes a 512×512 pre-disaster image and outputs a 512×512 single-channel mask predicting building presence per pixel.

**Approach:** built and verified incrementally, one component at a time, rather than writing the full architecture at once:
1. `EncoderBlock` — conv → relu → conv → relu → save skip connection → pool
2. `Bottleneck` — the deepest point of the U, same conv pattern but no pooling and no skip connection
3. `DecoderBlock` — upsample → concatenate with the matching skip connection → conv → relu → conv → relu
4. `UNet` — assembles 3 encoder blocks, 1 bottleneck, 3 decoder blocks, and a final 1×1 output convolution into one complete model

Each component was tested on a dummy input tensor immediately after being written, verifying its output shape matched hand-calculated expectations before moving to the next piece — the same "verify before trusting" discipline applied throughout this project.

**Design decision — parameterized channel depth.** Rather than hardcoding channel counts (64→128→256→512), the architecture uses a `base_channels` parameter (default 64, the standard convention from the original U-Net paper), so a lighter variant (e.g. `base_channels=32`) can be created for the Phase 4.5 multi-model comparison without duplicating the class.

**Placement:** the full architecture (`EncoderBlock`, `Bottleneck`, `DecoderBlock`, `UNet`) lives in `src/unet.py`, since it will be reused across Phase 5 (loading the trained model), Phase 6/8 (evaluation and benchmarking), and Phase 10 (deployment) — following the same `src/` vs. `notebooks/` split used throughout the project.

In [1]:
from pathlib import Path
import sys
sys.path.append(str(Path("..").resolve()))
import torch
from src.unet import UNet

dummy_input = torch.randn(1, 3, 512, 512)  # 1 fake image, 3 channels (RGB), 512x512
model = UNet()
output = model(dummy_input)
print("Final output shape:", output.shape)

Final output shape: torch.Size([1, 1, 512, 512])


### Observations

Each component's output shape was verified against hand-calculated expectations at every stage:

| Stage | Shape |
|---|---|
| Input | (1, 3, 512, 512) |
| After `block1` (pooled) | (1, 64, 256, 256) |
| After `block2` (pooled) | (1, 128, 128, 128) |
| After `block3` (pooled) | (1, 256, 64, 64) |
| After `bottleneck` | (1, 512, 64, 64) |
| After `decoder_block3` | (1, 256, 128, 128) |
| After `decoder_block2` | (1, 128, 256, 256) |
| After `decoder_block1` | (1, 64, 512, 512) |
| After `final_conv` (assembled `UNet`) | (1, 1, 512, 512) |

The assembled `UNet` class, imported from `src/unet.py`, reproduces the exact same verified output shape as the manually-chained component test — confirming the move to `src/` preserved the architecture correctly. The model correctly returns to the original 512×512 spatial resolution with a single output channel, matching the binary building/not-building prediction target established in Phase 3.3.

The architecture is complete and ready for training (Phase 4.3): the loss function, optimizer, and training loop, which will use this model together with the `SegmentationDataset` and `DataLoader` from Phase 3.

## 4.3 Loss Function, Optimizer, and Training Loop

**Task:** Set up the components that turn the U-Net architecture (4.2) into something that can actually learn — a loss function to measure prediction error, an optimizer to correct the model based on that error, and the training loop that ties them together with the `SegmentationDataset`/`DataLoader` pipeline.

**Loss function: `BCEWithLogitsLoss`.** Chosen because the task is per-pixel binary classification (building or not), and the model's `final_conv` layer outputs raw logits (unbounded real numbers), not probabilities. `BCEWithLogitsLoss` applies sigmoid internally and computes binary cross-entropy in one numerically stable step, rather than applying sigmoid and BCE as two separate operations.

**Optimizer: `Adam`**, given `model.parameters()` (every learnable weight in the network) and a learning rate of `1e-4` — a standard, well-tested starting choice for a first model, requiring little manual tuning.

**Training loop, per batch:** zero out old gradients → forward pass (get predictions) → compute loss against ground truth → backpropagate (compute gradients) → optimizer step (update weights). This cycle repeats for every batch, across every epoch.

**A real bug surfaced and fixed during this section:** `SegmentationDataset.__getitem__` was returning PIL Image objects rather than tensors — this went unnoticed in Phase 3.3's single-item testing (`dataset[0]`), since checking one example at a time doesn't require batching. `DataLoader`'s batching logic (`default_collate`) only accepts tensors, numpy arrays, or numbers — attempting to batch PIL Images directly raised a `TypeError`. Fixed by adding `torchvision.transforms.ToTensor()` conversion as the final step in `__getitem__`, after resizing and augmentation (which still require PIL Image inputs) — this also correctly adds the channel dimension needed for shape compatibility with the loss function.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from src.segmentation_dataset import SegmentationDataset

# Open the training and validation files
with open("../data/processed/train_ids.txt", "r") as f:
    train_ids = f.read().splitlines()

with open("../data/processed/val_ids.txt", "r") as f:
    val_ids = f.read().splitlines()

# Creating the batch for both train and validation
train_dataset = SegmentationDataset(train_ids, augment=True)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True) # batch of 8 examples

val_dataset = SegmentationDataset(val_ids, augment=False)  # no augmentation for validation
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

# Create the U-Net model
model = UNet()

# Create the loss function
criterion = nn.BCEWithLogitsLoss()

# Create the optimizer
optimizer = torch.optim.Adam(
    model.parameters(), # gives Adam access to all learnable weights in u-net
    lr=1e-4 # optimizer uses a learning rate of 0.0001 to determine the size of weight updates
)

# Put the model in training mode
model.train()

# Loop over batches from train_laoder
for batch_idx, (image, mask) in enumerate(train_loader):

    optimizer.zero_grad() # clear old gradients
    prediction = model(image) # get model's prediction (one logit for every pixel)
    loss = criterion(prediction, mask) # compute the loss
    loss.backward() # run the propagation (calculates the gradient for each learnable parameter)
    optimizer.step() # update the weights
    
    if batch_idx % 5 == 0:
        print(f"Batch {batch_idx}, Loss: {loss.item():.4f}")


Batch 0, Loss: 0.7508
Batch 5, Loss: 0.7308


### Observations

The training loop ran correctly end-to-end for the batches tested: `optimizer.zero_grad()` → `model(image)` → `criterion(prediction, mask)` → `loss.backward()` → `optimizer.step()`, with no shape or dtype errors after the `ToTensor()` fix — confirming the full pipeline (Dataset → DataLoader → model → loss → backpropagation → optimizer) is correctly wired together.

The first-batch loss (0.7508) is close to the theoretical random-guessing baseline for binary cross-entropy (≈0.693, i.e. -ln(0.5)) — the expected, healthy starting point for an untrained model with no learned information yet. This is a meaningful sanity check: a wildly different starting loss (very large, `NaN`, or suspiciously near-zero) would have signaled a bug in the setup rather than a genuinely fresh model.

The measured per-batch time (~2 minutes) makes local CPU training impractical for real experimentation — a single epoch would take ~3 hours, and meaningful training typically requires many epochs. **Rather than attempting to push through locally, training was stopped after confirming the pipeline works correctly**, and will resume on Colab/Kaggle GPU in Phase 4.4 — this local run's purpose (pipeline validation, not real training) was fully achieved before stopping.